In [1]:
# Библиотеки
from pyspark.sql import SparkSession
from datetime import datetime
import pyspark.sql.functions as F

In [2]:
!ls -la /opt/spark/jars/ | grep -E "hadoop-aws|aws-java-sdk|commons"

-rwxrwxrwx 1 root root 280645251 Feb 20 16:37 aws-java-sdk-bundle-1.12.262.jar
-rwxrwxrwx 1 root root    632424 Feb 20 16:37 commons-compress-1.20.jar
-rwxrwxrwx 1 root root    322780 Feb 20 16:53 commons-net-3.10.0.jar
-rwxrwxrwx 1 root root    962685 Feb 20 16:37 hadoop-aws-3.3.4.jar


In [3]:
spark = SparkSession.builder \
    .appName("Final_report") \
    .master("local[*]") \
    .config("spark.jars", 
            "/opt/spark/jars/clickhouse-jdbc-0.4.6.jar,"
           "/opt/spark/jars/hadoop-aws-3.3.4.jar,"
           "/opt/spark/jars/aws-java-sdk-bundle-1.12.262.jar,"
           "/opt/spark/jars/commons-net-3.10.0.jar,"
           "/opt/spark/jars/commons-compress-1.20.jar") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "StrongPassword123!") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ SparkSession готова", datetime.now())

✅ SparkSession готова 2026-02-26 10:00:41.143456


In [4]:
df_test = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:clickhouse://clickhouse:8123/mart") \
    .option("dbtable", "(SELECT * FROM clean_purchases) AS subq") \
    .option("user", "default") \
    .option("password", "password") \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .load()
# перенос DF  в оперативную память 
df = df_test.cache()

In [5]:
# вывод схемы 
df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- purchase_id: string (nullable = true)
 |-- purchase_datetime: timestamp (nullable = true)
 |-- product_id: string (nullable = true)
 |-- group: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- total_amount: decimal(10,2) (nullable = true)
 |-- is_organic: integer (nullable = true)
 |-- is_delivery: integer (nullable = true)
 |-- is_loyalty_member: integer (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- registration_date: timestamp (nullable = true)



In [6]:
# Уникальные значения в столбце к ним буду проводить left join
df_cus_uniq = df.select("customer_id").distinct()
df_cus_uniq.show(3)

+-----------+
|customer_id|
+-----------+
|   cus-1035|
|   cus-1007|
|   cus-1006|
+-----------+
only showing top 3 rows



In [7]:
# 🥛 пользоватля покупавшие молоко за последние 30 дней 
bought_milk_last_30d = df.filter(
    (F.col("group") == "Молочные продукты") & 
    (F.col("purchase_datetime") >= F.date_sub(F.current_date(), 30))
).select("customer_id").distinct()

In [8]:
#🍏 Покупалb фрукты и ягоды за последние 14 дней
bought_fruits_last_14d = df.filter(
    (F.col("group") == "Фрукты и ягоды") &
    (F.col("purchase_datetime") >= F.date_sub(F.current_date(), 14))
).select("customer_id").distinct()

In [9]:
#🥦 Не покупал овощи и зелень за последние 14 дней
not_bought_veggies_14d = df.filter(
    (F.col("group") != "Овощи и зелень") &
    (F.col("purchase_datetime") >= F.date_sub(F.current_date(), 14))
).select("customer_id").distinct()

In [10]:
# Делал более 2 покупок за последние 30 дней
recurrent_buyer = (df
    .filter(F.col("purchase_datetime") >= F.date_sub(F.current_date(), 30)) \
    .groupBy("customer_id") \
    .agg(F.count("purchase_id").alias("cnt_pur")) \
    .filter(F.col("cnt_pur") >= 2).select("customer_id")
)


In [11]:
# Не покупал 14–30 дней (ушедший клиент?)
inactive_14_30 = (df
    .groupBy("customer_id")
    .agg(F.max("purchase_datetime").alias("last_pur"))
    .filter(
        (F.col("last_pur") <= F.date_sub(F.current_date(), 14)) &
        (F.col("last_pur") > F.date_sub(F.current_date(), 30)))
    .orderBy(F.col("last_pur"))\
    .select("customer_id")
    )


In [12]:
#Покупатель зарегистрировался менее 30 дней назад
new_customer = df.filter(
    F.col("registration_date") >= F.date_sub(F.current_date(), 30))\
.select("customer_id").distinct()


In [13]:
# Пользовался доставкой хотя бы раз
delivery_user = df.filter(
    F.col("is_delivery") == 1)\
.select("customer_id").distinct()

In [14]:
# Купил хотя бы 1 органический продукт
organic_preference = df.filter(
    F.col("is_organic") == 1)\
.select("customer_id").distinct()

In [15]:
# Средняя корзина > 1000₽
bulk_buyer = (df
    .groupBy("customer_id") \
    .agg(F.avg("total_amount").alias("avg_amount")) \
    .filter(F.col("avg_amount") > 1000)\
.select("customer_id")
    )


In [16]:
# Средняя корзина < 200₽
low_cost_buyer = (df
    .groupBy("customer_id") \
    .agg(F.avg("total_amount").alias("avg_amount")) \
    .filter(F.col("avg_amount") < 1000)\
.select("customer_id")
    )

In [17]:
# Покупал хлеб/выпечку хотя бы раз
buys_bakery = df.filter(
    F.col("group") == "Зерновые и хлебобулочные изделия")\
.select("customer_id").distinct()


In [18]:
# Финальный результат для отчета
result = df_cus_uniq.alias("all_cust")\
.join(
    bought_milk_last_30d.alias("milk"), 
    F.col("all_cust.customer_id") == F.col("milk.customer_id"), 
    "left")\
.join(
    bought_fruits_last_14d.alias("fruits"),
    F.col("all_cust.customer_id") == F.col("fruits.customer_id"), 
    "left") \
.join(
    not_bought_veggies_14d.alias("not_vegg"),
    F.col("all_cust.customer_id") == F.col("not_vegg.customer_id"), 
    "left") \
.join(
    recurrent_buyer.alias("rec_buyer"),
    F.col("all_cust.customer_id") == F.col("rec_buyer.customer_id"), 
    "left") \
.join(
    inactive_14_30.alias("inact"),
    F.col("all_cust.customer_id") == F.col("inact.customer_id"), 
    "left") \
.join(
    new_customer.alias("new_cus"),
    F.col("all_cust.customer_id") == F.col("new_cus.customer_id"), 
    "left") \
.join(
    delivery_user.alias("deliv_user"),
    F.col("all_cust.customer_id") == F.col("deliv_user.customer_id"), 
    "left") \
.join(
    organic_preference.alias("organic"),
    F.col("all_cust.customer_id") == F.col("organic.customer_id"), 
    "left") \
.join(
    bulk_buyer.alias("b_b"),
    F.col("all_cust.customer_id") == F.col("b_b.customer_id"), 
    "left") \
.join(
    low_cost_buyer.alias("low_cost"),
    F.col("all_cust.customer_id") == F.col("low_cost.customer_id"), 
    "left") \
.join(
    buys_bakery.alias("bakery"),
    F.col("all_cust.customer_id") == F.col("bakery.customer_id"), 
    "left") \
.select(                                              
        F.col("all_cust.customer_id"),
        F.when(F.col("milk.customer_id").isNotNull(), 1).otherwise(0).alias("bought_milk_last_30d"),
        F.when(F.col("fruits.customer_id").isNotNull(), 1).otherwise(0).alias("bought_fruits_last_14d"),
        F.when(F.col("not_vegg.customer_id").isNotNull(),1).otherwise(0).alias("not_bought_veggies_14d"),
        F.when(F.col("rec_buyer.customer_id").isNotNull(),1).otherwise(0).alias("recurrent_buyer"),
        F.when(F.col("inact.customer_id").isNotNull(),1).otherwise(0).alias("inactive_14_30"),
        F.when(F.col("new_cus.customer_id").isNotNull(),1).otherwise(0).alias("new_customer"),
        F.when(F.col("deliv_user.customer_id").isNotNull(),1).otherwise(0).alias("delivery_user"),
        F.when(F.col("organic.customer_id").isNotNull(),1).otherwise(0).alias("organic_preference"),
        F.when(F.col("b_b.customer_id").isNotNull(),1).otherwise(0).alias("bulk_buyer"),
        F.when(F.col("low_cost.customer_id").isNotNull(),1).otherwise(0).alias("low_cost_buyer"),
        F.when(F.col("bakery.customer_id").isNotNull(),1).otherwise(0).alias("buys_bakery")
    ) \
.orderBy("all_cust.customer_id")


result.show(1, vertical= True)

-RECORD 0--------------------------
 customer_id            | cus-1000 
 bought_milk_last_30d   | 0        
 bought_fruits_last_14d | 0        
 not_bought_veggies_14d | 1        
 recurrent_buyer        | 0        
 inactive_14_30         | 0        
 new_customer           | 1        
 delivery_user          | 1        
 organic_preference     | 1        
 bulk_buyer             | 0        
 low_cost_buyer         | 1        
 buys_bakery            | 1        
only showing top 1 row



In [19]:
# сохранение отчета в CSV и parquet
result.write.mode("overwrite").csv("s3a://report-clickhouse/report.csv")
print("✅ ЗАПИСЬ ОТЧЕТА CSV УСПЕШНА!")

✅ ЗАПИСЬ ОТЧЕТА CSV УСПЕШНА!


In [20]:
# сохранение отчета в CSV и parquet
result.write.mode("overwrite").parquet("s3a://report-clickhouse/report.parquet")
print("✅ ЗАПИСЬ ОТЧЕТА PARQUET УСПЕШНА!")

✅ ЗАПИСЬ ОТЧЕТА PARQUET УСПЕШНА!


In [21]:
spark.stop()
print("✓ SparkSession остановлена", datetime.now())

✓ SparkSession остановлена 2026-02-26 10:00:57.001293
